In [ ]:
#
# This notebook is for the exploration of Logistic Regression -- it corresponds to handout-02
# https://frcs.github.io/4C16-LectureNotes/logistic-regression.html
#

# 📘 Lab 2 study guide: Logistic regression and gradient descent (handout-02)

**Big picture.** Lab 1 predicted a *number*. Lab 2 predicts a *class* (default: yes/no; iris species). You'll learn three ideas that carry over to every neural network in Labs 4–5:
1. **Model a probability** with a squashing function: the sigmoid (2 classes) or softmax (K classes).
2. **Train with the cross-entropy loss**, the negative log-likelihood of the labels.
3. **Optimise by gradient descent**, because there is no closed-form solution any more.

**Concept map (handout-02 slide numbers)**

| Concept | Handout-02 | In this notebook |
|---|---|---|
| Why least squares is poor for classification | slides 6–10 | motivation |
| Logit / risk score $h_{\mathbf w}(\mathbf x)=\mathbf x^\top\mathbf w$ | slides 11–14 | `lab_2.logit` |
| Sigmoid $\sigma(t)=1/(1+e^{-t})$, $p(y=1\mid\mathbf x)=\sigma(\mathbf x^\top\mathbf w)$ | slides 18–19 | `lab_2.predict` |
| Likelihood → cross-entropy loss | slides 23–24 | **Exercise 1** |
| Why not MSE for classification | slide 25 | note after Ex 1 |
| Gradient descent and the learning rate | slides 26–28 | `gradient_descent`, **Exercise 3** |
| Gradient of the cross-entropy, $\frac1n X^\top(\mathbf p-\mathbf y)$ | slides 29–30 | **Exercise 2** |
| Decision threshold → class, accuracy | slides 19, 32; handout-04 slides 6–13 | **Exercises 4–5** |
| Softmax / multinomial logistic regression | slides 35–38 | **Exercises 6–7** |

**Exercise checklist** (everything goes in `lab_2.py`; the local file is still the unsolved starter):

| # | Function | What to return | Verified value |
|---|---|---|---|
| 1 | `cross_entropy(w,X,y)` | average binary cross-entropy | test prints `0.632853327993248` |
| 2 | `gradient(w,X,y)` | $\frac1n X^\top(\sigma(X\mathbf w)-\mathbf y)$ | test prints `[0.06183342 0.29934383]` |
| 3 | `question_3()` | a learning rate that converges fastest | ≈ 20–25 (see note) |
| 4 | `predict_class(w,X,t)` | `1` if $p\ge t$ else `0` | used by the accuracy cell |
| 5 | `question_5()` | accuracy at $t=0.75$ | **0.9698** |
| 6 | `question_6()` | intercept of class 0 | **≈ 8.68** (`log_reg.intercept_[0]`) |
| 7 | `question_7()` | best feature pair and its accuracy | **`{'Features': (0, 2), 'Acc': 0.9556}`** |

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/4c16-labs/code/dist/lab-02/

In [ ]:
# Autoload setup (you don't need to edit this cell); instructions to: 
import importlib
import lab_2

In [ ]:
##############################################################
#
# 1: import
#
##############################################################

import pandas as pd        # for datasets
import numpy as np         # for linear algebra
from tqdm import tqdm      # for progress bars 

import matplotlib as mpl   # for plotting graphs
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import preprocessing

%matplotlib inline


In [ ]:
# sets default style for graphs

plt.style.use('ggplot')
plt.rcParams['lines.linewidth'] = 1.7
plt.rcParams["figure.figsize"] = (10,6)


In [ ]:
##############################################################
#
# 2. loading the ISLR 'Default' dataset
#
##############################################################

# see https://cran.r-project.org/web/packages/ISLR/ISLR.pdf
#
# This data set contains information on ten thousand customers. 
#
# The aim here is to predict which customers will default on their credit card debt.
#
# The dataset contains 10000 observations on the following 4 variables.
#   * 'default': a No/Yes label indicating whether the customer defaulted on their debt
#   * 'student': a No/Yes label indicating whether the customer is a student
#   * 'balance': the average balance that the customer has remaining on their credit card after making
#                their monthly payment
#   * 'income' : income of customer

df = pd.read_csv('Default.csv')

# we are using here the pandas python package to read the CSV file.
# we can look at the first 10 observations

df.head(10)

### 💡 Note: the `Default` dataset

10,000 credit-card customers, with features `balance` and `income` (plus `student`) and a binary label `default`. Only **333 (3.33%)** defaulted, so the classes are heavily **imbalanced**. Keep this in mind: a "classifier" that always says *No* is already 96.7% accurate. Exercise 5 comes back to this, and Lab 3 / handout-04 deal with it properly using precision, recall and ROC curves.

In [ ]:
##############################################################
#
# 3. visualise your data
#
##############################################################

# We need to visualise our data
# let's ignore the categorical features 'student' and the first column
# and focus on the 'balance' and 'income' features

balance = df['balance'].values
income = df['income'].values

# the outcome is of boolean type, converting it to vector of integers
y = (df['default'].values == 'Yes').astype(int)

# we are only going to plot a subset of the data
income_subset = income[0:1000];
balance_subset = balance[0:1000];
y_subset = y[0:1000];

# plotting balance vs income for the 'No Default' class
plt.scatter(balance_subset[y_subset == 0],
            income_subset[y_subset == 0], 
            s=15, marker='o')

# plotting balance vs income for the 'Default' class
plt.scatter(balance_subset[y_subset == 1],  
            income_subset[y_subset == 1],  
            s=40, marker='+')

plt.ylim(ymin=0)
plt.ylabel('Income')
plt.xlim(xmin=-100)
plt.xlabel('Balance')
plt.legend(['No Default', 'Default'])
plt.show()

# after a quick look at the graph, it appears that 
# the most relevant feature is 'balance'


### 🔍 Interpreting the scatter plot

Defaulters (`+`) are concentrated at **high balance**, at every income level. The classes are separated mostly along the *balance* axis, so the notebook uses a single feature:
$$p(\text{default}\mid\text{balance}) = \sigma(w_0 + w_1\cdot\text{balance}).$$
The design matrix is $X=[\mathbf 1,\ \text{balance}]$. The column of ones carries the bias $w_0$, exactly as the $x^0$ column did in Lab 1.

In [ ]:
# Exercise 1: implement cross-entropy (in the lab_2 module)

# Import lab_2 again
importlib.reload(lab_2)

w_test = np.array([[0.1], [0.3]])
X_test = np.array([[1,1], [1,0], [3,2]])
y_test = np.array([[0], [1], [1]])
print(lab_2.cross_entropy(w_test, X_test, y_test))

# Should print 0.632853327993248 if your function is working.

### Exercise 1: Average Cross-Entropy
**Objective.** Implement `lab_2.cross_entropy(w, X, y)` to measure the average error of the binary logistic-regression model.

`predict(w, X)` should first produce probabilities $p_i = \sigma(x_i^\top w)$. The required loss is the binary cross-entropy
$$\mathcal{L}(w)=-\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(p_i+\varepsilon)+(1-y_i)\log(1-p_i+\varepsilon)\right],$$
where the small $\varepsilon$ prevents taking $\log(0)$. This loss penalises confident incorrect predictions strongly and is the objective minimised during training. The notebook calls the function on a small fixed example and checks that the result is approximately `0.632853327993248`; gradient descent later records the same loss to assess optimisation progress.

### 🛠️ How to do Exercise 1, step by step (handout-02 slides 23–24)

**Where the loss comes from.**
- Each label is a Bernoulli draw with probability $p_i=\sigma(\mathbf x_i^\top\mathbf w)$, so the likelihood of the data is $\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$.
- Taking $-\log$ and averaging gives the **average cross-entropy**:
$$\mathcal L(\mathbf w)=-\frac1n\sum_{i=1}^n\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big].$$
- Minimising the cross-entropy = **maximising the likelihood**, the same principle that gave MSE in Lab 1.

**Code** (vectorised; `eps` avoids $\log 0$):
```python
def cross_entropy(w, X, y):
    p = predict(w, X)                       # probabilities sigma(Xw), same shape as y
    eps = 0.000001
    return np.mean(-y * np.log(p + eps) - (1 - y) * np.log(1 - p + eps))
```
**Check by hand** for the test (w = [0.1, 0.3]):
- logits: $[0.4, 0.1, 0.9]$
- $p = [0.599, 0.525, 0.711]$
- $\mathcal L=\frac13[-\log(1-0.599)-\log 0.525-\log 0.711]=\frac13[0.913+0.644+0.341]=0.6329$ ✔

**Why not MSE? (slide 25)** With a sigmoid output, MSE is non-convex in $\mathbf w$ and has tiny gradients when the model is confidently wrong. The cross-entropy is convex for logistic regression and punishes confident mistakes heavily ($-\log p\to\infty$ as $p\to0$). The same loss, in its multi-class form, is `nn.CrossEntropyLoss` in Labs 4–5.

In [ ]:
# Exercise 2: implement gradient computation (in the lab_2 module)
# Import lab_2 again
importlib.reload(lab_2)


w_test = np.array([0.1, 0.3])
X_test = np.array([[1,1], [1,0]])
y_test = np.array([0, 1])
print(lab_2.gradient(w_test, X_test, y_test))
# Expected result: [ 0.06183342  0.29934383]

### Exercise 2: Cross-Entropy Gradient
**Objective.** Implement `lab_2.gradient(w, X, y)` for the average cross-entropy objective.

For logistic regression, with $p=\sigma(Xw)$, the gradient is
$$\nabla\mathcal{L}(w)=\frac{1}{n}X^\top(p-y).$$
The implementation should obtain `p` with `predict`, form the residual $p-y$, and use `np.dot` to combine all observations efficiently. This vector gives the direction in which loss increases, so gradient descent updates $w \leftarrow w-\eta\nabla\mathcal{L}(w)$. The notebook tests the function against the expected two-component gradient, then uses it at every optimisation iteration.

### 🛠️ How to do Exercise 2, step by step (handout-02 slides 29–30)

**Derivation (worth knowing for exams).** Using $\sigma'(t)=\sigma(t)(1-\sigma(t))$, the per-sample loss $\ell_i=-y_i\log p_i-(1-y_i)\log(1-p_i)$ has
$$\frac{\partial\ell_i}{\partial\mathbf w}=(p_i-y_i)\,\mathbf x_i\quad\Rightarrow\quad\nabla\mathcal L=\frac1n\sum_i(p_i-y_i)\mathbf x_i=\frac1n X^\top(\mathbf p-\mathbf y).$$
The intuition: each sample pushes $\mathbf w$ along its own feature vector, in proportion to its prediction error $p_i-y_i$.

**Code** (the starter already divides by `n`):
```python
def gradient(w, X, y):
    n = y.shape[0]
    p = predict(w, X)
    grad = np.dot(X.T, p - y)
    return grad / n
```
**Check.**
- $p=\sigma([0.4,\,0.1])=[0.5987,\,0.5250]$, so $\mathbf p-\mathbf y=[0.5987,\,-0.4750]$.
- $X^\top(\mathbf p-\mathbf y)=[0.1237,\,0.5987]$; dividing by 2 gives `[0.06183342 0.29934383]` ✔

**Shape tip.** `w` and `y` must both be 1-D `(n,)` or both be `(n,1)` columns. Mixing them broadcasts `p - y` into an `(n,n)` matrix.

In [ ]:
# The gradient descent algorithm as explained in lecture notes
# the learning_rate refers to the greek letter 'eta'
# The method also returns the vectors of 
#   loss function, sampled iterations, weights
# Import lab_2 again
importlib.reload(lab_2)

def gradient_descent(w0, X, y, learning_rate=0.01, maxit=1000):
    print("starting gradient descent")
    w = w0
    n = X.shape[0]
    p = X.shape[1]
    step = 83 # interval between measures
    m = (maxit // step) + 1 # nb of checks
    loss = np.zeros((m,1))
    its = np.zeros((m,1))
    ws = np.zeros((m,p))

    for i in tqdm(range(0, maxit)):
        # monitoring the loss at regular intervals      
        if (i % step == 0):
            j = i // step
            loss[j] = lab_2.cross_entropy(w, X, y)
            its[j] = i
            ws[j,0:p] = w.ravel()
            
        # computing the loss gradient
        grad = lab_2.gradient(w,X,y);
 
        # updating the weights
        w = w - learning_rate * grad;        
        i = i + 1
    return w, loss, its, ws
        

In [ ]:
#
# Setting up the Model and Design Matrix
#
# in this lab we will look at the following model:
# y = [ w0 + w1 * balance > 0]
#
# and ignore the 'income' feature

n = balance.shape[0]  # number of observations
p = 2                 # number of features

X = np.zeros(shape=(n, p))
print(X.shape)

# the first feature is 1 (with associated weight w0)
X[:,0] = 1;

# the second feature is the balance values (with associated weight w1)
X[:,1] = balance[:]
print(X.shape)

# initial weights
w0 = np.array([0,0]);

# starting gradient descent optimisation with some default parameters
w, loss, its, ws = gradient_descent(w0, X, y, learning_rate=40, maxit=5000);
print(w)

# plot of the loss graph, ie.
plt.plot(its, loss)
plt.title("Loss Graph")
plt.xlabel('gradient descent iterations')
plt.ylabel('average cross-entropy')
plt.show()


In [ ]:
# if you look carefully, you will find that the gradient descent is not working 
# very well. You can see this because the overall loss is not going down, oscillates 
# and never gets low enough (eg. avg cross-entropy > .5, when we would like something < 0.1)

### 💡 Concept note: gradient descent (handout-02 slides 26–28)

$$\mathbf w^{(k+1)}=\mathbf w^{(k)}-\eta\,\nabla\mathcal L\big(\mathbf w^{(k)}\big)$$
- $-\nabla\mathcal L$ points downhill. The learning rate $\eta$ (`learning_rate`) sets the step length.
- `gradient_descent` records the loss and the weights every 83 iterations so you can plot them.
- The **first run uses raw `balance` (0–2650)** with $\eta=40$. The gradient component for $w_1$ is about 1000× larger than for $w_0$, so the steps in $w_1$ are enormous. The loss oscillates and never gets below ~0.5. This is an *ill-conditioned* problem: one learning rate cannot suit both weights.

In [ ]:
# to better study the convergence issues, we'll also plot the weights 
# we encapsulate all these plots a single function:

def plot_loss_graph_and_weight_evolution(loss, ws, its):

  f, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2)

  f.set_figheight(15)
  f.set_figwidth(15)

  ax1.plot(its, loss)
  ax1.set_xlabel('gradient descent iterations')
  ax1.set_ylabel('average cross-entropy')

  ax2.plot(ws[:,0], ws[:,1], 
          color='g', marker='o', 
          linestyle='dashed', linewidth=0.5, markersize=4)
  ax2.set_title('Evolution of the Weights')
  ax2.set_xlabel('$w_0$')
  ax2.set_ylabel('$w_1$')

  ax3.plot(its, ws[:,0], 'g-')
  ax3.set_title('Evolution of $w_0$')
  ax3.set_xlabel('gradient descent iterations')
  ax3.set_ylabel('$w_0$')

  ax4.plot(its, ws[:,1], 'g-')
  ax4.set_title('Evolution of $w_1$')
  ax4.set_xlabel('gradient descent iterations')
  ax4.set_ylabel('$w_1$')
  
  plt.show()

In [ ]:
# plotting the evloution of loss + weights

plot_loss_graph_and_weight_evolution(loss, ws, its)

# we can see that the weights are diverging



In [ ]:
# One problem here is that Gradient Descent algorithms are quite sensitive 
# to initial conditions and it is often beneficial to rescale or offset 
# the features to improve the performance. 
#
# In our case we note that balance ranges from 0 to about 2500.
# this is much larger in magnitude to the first feature which is simply 1.
# Thus we (arbitrarily) rescale by factor of 1/1000

X[:,1] = balance[:]/1000;

# note to self: we'll need to apply this rescaling  
# everytime we are making predictions.

# starting again Gradient Descent with same parameters
w, loss, its, ws = gradient_descent(w0, X, y, learning_rate=40, maxit=5000);
print(w)

# plot of the loss graph
plot_loss_graph_and_weight_evolution(loss, ws, its)



In [ ]:
# we can see that we are doing much better, with the loss going down to about an
# avg cross entropy of 0.1, but it still seems to be oscillating,
# which indicates that the learning rate is probably too high (see lecture notes)

In [ ]:
# note that you can also check if you reached convergence by checking if the 
# gradient is null (or at least very small):
# Import lab_2 again
importlib.reload(lab_2)

print(lab_2.gradient(w, X, y))

### 🔍 Interpreting the result: why rescaling fixed it

Dividing balance by 1000 puts both features on a similar scale (≈ 1). The loss surface becomes rounder, so a single $\eta$ works for both weights, and the loss falls to ≈ 0.08.

**General rule:** standardise or rescale input features before gradient-based training. Lab 3 uses `StandardScaler`, and Labs 4–5 divide pixels by 255.

⚠️ After rescaling, the learned $w_1$ is "per £1000 of balance". Any new data must be rescaled the same way before prediction.

The gradient-check cell then prints $\nabla\mathcal L(\mathbf w)$. At a true minimum it should be ≈ 0, which is a cheap convergence test (compare the Newton stopping rule in Lab 0).

In [ ]:
# Exercise: study the convergence for a few different learning rates, 
# and observe the different behaviours.
# for instance you can try lr values in {150, 30, 10, 1, 0.001}

lr = 150 # change learning rate here
w0 = np.array([0,0]);
print(f"learning rate = {lr}")
w, loss, its, ws = gradient_descent(w0, X, y, learning_rate=lr, maxit=3000);
plot_loss_graph_and_weight_evolution(loss, ws, its)

# Question 3 
# Find the learning rate that allows us to converge the fastest to the global optimum.
# eg. lr = 45 (doesn't have to be super precise, push to assessment to check if answer is ok)
# write answer in question_3 of lab_2.py


### Exercise 3: Learning Rate and Convergence
**Objective.** Experiment with the learning rate `lr` and report in `lab_2.question_3()` a value that reaches the global optimum quickly.

The update $w_{k+1}=w_k-\eta\nabla\mathcal{L}(w_k)$ depends on the step size $\eta$. A small value usually converges slowly, whereas a large value can overshoot the minimum and cause oscillation or divergence. The preceding feature rescaling makes the optimisation better conditioned, but `lr` still controls stability and speed. This cell trains from the same initial weights for several values and plots loss and weight trajectories; the best practical rate is the one with a rapid, stable decrease in cross-entropy.

### 🛠️ How to do Exercise 3 (choosing the learning rate)

**Procedure**
1. Set `lr` in the cell above, run it, and look at the loss curve and the weight trajectory.
2. Classify the behaviour:
   - **too small**: smooth but slow, not converged after 3000 iterations
   - **good**: fast, smooth drop to the plateau ≈ 0.080, with weights settling at $\mathbf w^\star\approx[-10.65, 5.50]$
   - **too large**: zig-zag or oscillating weights, a loss that bounces or stalls above the minimum, or divergence
3. Pick the **largest rate that still converges smoothly**. That is the fastest.

**Verified behaviour** (rescaled data, start at $\mathbf w=[0,0]$, 3000 iterations, "reached" = within 0.05 of $\mathbf w^\star$):

| lr | outcome |
|---|---|
| 0.001 | barely moves (w ≈ [−0.86, −0.58]) |
| 1 | slow: still short of the optimum after 3000 iterations |
| 10 | converges (≈ 770 iterations) |
| 20 | converges (≈ 380 iterations) |
| 25 | converges fastest (≈ 260 iterations) |
| ≥ 26 (e.g. 30, 40) | oscillates and does not settle within 3000 iterations |
| 150 | diverges / oscillates wildly |

So a sensible answer for `question_3()` is a value around **20–25**. The comment in the code says it "doesn't have to be super precise", so push to the autograder to confirm your choice is within tolerance.

**Concept.** For a quadratic-like bowl, gradient descent is stable only if $\eta<2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest curvature (Hessian eigenvalue). Close to that limit it converges fastest; above it, it diverges. Labs 4–5 handle this automatically with **learning-rate schedules** (cosine annealing) and **adaptive optimisers** (Adam) (handout-05 slides 50–52, handout-07 slides 31–32).

In [ ]:
# In logistic Regression, we set a parametric model for the likelihood 
# we denote logit = x'w and parametrise the likelihood as
# p(y_i=1|logit) = 1/(1 + exp(-logit))
#
# We want to verify that this is a correct approximation for our problem
#
# The following function makes an empirical measurement of p(y_i=1|logit)
# by recording in the dataset the proportion of default=True for 
# a particular logit value (within some small threshold T=1).

# Import lab_2 again
importlib.reload(lab_2)

def get_empirical_likelihood(logit, logits_train, y):  
    # selecting all logits in training set that are +/- 0.5 of given logit
    valid = ((logits_train < logit + 0.5) & (logits_train > logit - 0.5));    
    n_positives = sum(valid[y==True]);
    n_negatives = sum(valid[y==False]);
    empirical_likelihood = n_positives / (n_positives  + n_negatives);
    return empirical_likelihood


# taking an ok value for w:
w = np.array([-10.63971053,   5.49188453])  

# testing set consists of regularly sampled 100 values of balance
n_test = 100
X_test = np.zeros(shape=(n_test,2))
X_test[0:n_test,0] = 1;
X_test[0:n_test,1] = np.linspace(X[:,1].min(), X[:,1].max(), num=n_test)

# making predictions for these values
p_test = lab_2.predict(w, X_test)

# we compute the logit values and their corresponding empirical probabilities of default
logits_test = lab_2.logit(w, X_test)
logits_train = lab_2.logit(w, X)
p_empirical = [get_empirical_likelihood(logit, logits_train, y) 
                  for logit in logits_test ];

# plot the graphs

plt.scatter(X[y==False,1], y[y==False], alpha= 0.2)
plt.scatter(X[y==True,1], y[y==True], alpha= 0.2)
plt.plot(X_test[:,1], p_test, color='black')
plt.plot(X_test[:,1], p_empirical, ':', color='gray')

plt.ylabel('Probability of default');
plt.xlabel('Balance');
plt.yticks([0, 0.25, 0.5, 0.75, 1.]);
plt.legend(['logistic model for probability of default',
            'empirically measured probability of default\n(within a +/- 0.5 logit bin)', 
            'No Default', 'Default'],  prop={'size': 12})
plt.show()

# you should be able to see that the logistic model is not a bad approximation 
# of the empirical likelihood


### 🔍 Interpreting the result: is the logistic model a good fit?

The black curve is the model $p(\text{default}\mid\text{balance})=\sigma(-10.64+5.49\cdot\text{balance}/1000)$. The dotted grey curve is the fraction of actual defaulters among customers with a similar logit. Where they agree, the model's probabilities are **calibrated**: when it says 30%, about 30% of those customers really default.

Reading the curve: the probability crosses 0.5 at $\text{balance}=10.64/5.49\times1000\approx$ **£1,940**. That is the decision boundary for threshold 0.5.

In [ ]:
# Import lab_2 again
importlib.reload(lab_2)

# At this point, we still don't have a classifier. 
# All we need is to set a threshold on the predicted probabilities

# #### EXERCISE 4 ####
# In lab_2.py, write a function 'predict_class' in the module to give
# the predicted class for observations X and weights w.

# Use that function here to assess the accuracy of the classifier
# for different thresholds.

# Accuracy = percentage correctly classified.
def accuracy(w, X, y, threshold):
    return np.mean(y == lab_2.predict_class(w, X, threshold))

# taking an ok value for w:
w = np.array([-10.63971053,   5.49188453])  

print("Accuracy for T=0.25: {}".format(accuracy(w, X, y, threshold=0.25)))
print("Accuracy for T=0.50: {}".format(accuracy(w, X, y, threshold=0.5)))
print("Accuracy for T=0.75: {}".format(accuracy(w, X, y, threshold=0.75)))
print("Accuracy for T=0.95: {}".format(accuracy(w, X, y, threshold=0.95)))

# #### EXERCISE 5 ####
# in lab_2.py, update function 'question_5' to 
# return the accuracy for a threshold of 0.75

### Exercises 4 and 5: Thresholded Classification and Accuracy
**Exercise 4 objective.** Implement `lab_2.predict_class(w, X, t)` by converting each predicted probability into a class label, for example $\hat{y}_i=\mathbb{1}[\sigma(x_i^\top w)\geq t]$. The threshold $t$ expresses the trade-off between predicting defaults and avoiding false positives; it is not determined by probability estimation alone.

**Exercise 5 objective.** Update `lab_2.question_5()` to return the accuracy at $t=0.75$, where
$$\operatorname{accuracy}=\frac{1}{n}\sum_{i=1}^{n}\mathbb{1}[y_i=\hat{y}_i].$$
The notebook’s `accuracy` helper compares the implementation’s labels with the known outcomes at several thresholds. This demonstrates how an evaluation metric changes with the decision rule and why accuracy alone can be limited for an imbalanced default dataset.

### 🛠️ How to do Exercises 4 and 5

**Exercise 4: `predict_class`.** Compare the predicted probability with the threshold and return 0/1 labels with the **same shape as `y`**:
```python
def predict_class(w, X, t):
    return (predict(w, X) >= t).astype(int)
```
⚠️ The starter returns `np.zeros((n,1))`, which has shape `(n,1)`. The notebook's `accuracy` compares with `y` of shape `(n,)`, so an `(n,1)` result broadcasts to `(n,n)` and gives a meaningless number. Returning `predict(w, X) >= t` (shape `(n,)` because `w` is 1-D) avoids this.

**Exercise 5.** Read the printed accuracy for `T=0.75` and return it:
```python
def question_5():
    return 0.9698
```

**Verified accuracies** (w = [−10.6397, 5.4919]): T=0.25 → 0.9653, T=0.50 → **0.9725**, T=0.75 → **0.9698**, T=0.95 → 0.9671.

**Interpretation: why accuracy is misleading here (handout-04 slides 6–17).**
- All four values are close to **0.9667**, which is the accuracy of always predicting *No*.
- Raising $t$ makes the model predict *default* less often: fewer false positives, but more missed defaulters (false negatives).
- The threshold should reflect the *costs* of the two errors. A bank would rather flag a few extra customers than miss defaulters, which argues for a low $t$.
- Lab 3 measures this trade-off properly with precision/recall, F1 and ROC curves.

# Multinomial Logistic Regression



In [ ]:
# Let's now see how to work with more than 2 classes.
# loading the Iris dataset.
# This is a foumous 3-classes dataset, from Fisher, 1936

from sklearn.datasets import load_iris
iris = load_iris()
print(iris['DESCR'])


In [ ]:
# let's explore the dataset
# as we have 4 features, we will break down the visualisation into pairs of 
# features

X = iris.data
Y = iris.target
class_names = iris.target_names
feature_names = iris.feature_names

df = pd.DataFrame(X, columns=iris.feature_names)

grr = pd.plotting.scatter_matrix(df, c=Y, figsize=(12, 12),grid=True, 
                                 hist_kwds={'bins': 20}, s=60, alpha=.8)


In [ ]:
# we are going to split the dataset into training and test tests in a 70/30 ratio

trainX, testX, trainY, testY = train_test_split(X, Y, test_size = 0.3, random_state = 11)


In [ ]:
# this time we'll use off-the-shelf functions to get the optimisation

log_reg = LogisticRegression(solver='newton-cg', multi_class='multinomial',random_state = 11)
log_reg.fit(trainX, trainY)
y_pred = log_reg.predict(testX)

In [ ]:
print('Accuracy:   {:.2f}'.format(np.mean(testY ==  y_pred)))
print('Error rate: {:.2f}'.format(1 - np.mean(testY ==  y_pred)))


### 💡 Concept note: multinomial (softmax) logistic regression (handout-02 slides 35–38)

With $K$ classes, each class $k$ gets its own score (logit) $s_k(\mathbf x)=b_k+\mathbf x^\top\mathbf w_k$. These are turned into probabilities with the **softmax**:
$$P(y=k\mid\mathbf x)=\frac{e^{s_k(\mathbf x)}}{\sum_{j=1}^{K}e^{s_j(\mathbf x)}},\qquad \mathcal L=-\frac1n\sum_i\log P(y=y_i\mid\mathbf x_i).$$
- For $K=2$ this reduces to the sigmoid model.
- This is **exactly** the last layer and loss of the CIFAR-10 networks in Labs 4–5: the network outputs 10 logits, and `nn.CrossEntropyLoss` applies the softmax and the negative log-likelihood.

**Iris:** 150 flowers, 4 features (sepal length/width, petal length/width, in cm), 3 species. In the scatter matrix, *setosa* separates cleanly using the **petal** features, while *versicolor* and *virginica* overlap a little.

**Split:** 70/30 train/test with `random_state=11`. Keep it fixed so that your answers match the autograder.

**Fit:** `LogisticRegression(solver='newton-cg', multi_class='multinomial')` uses Newton's method (Lab 0!) on the softmax cross-entropy. sklearn adds L2 regularisation by default (`C=1.0`). The reported **test accuracy is 0.93** (verified: 0.9333 = 42/45).

⚠️ `multi_class=` is deprecated and has been removed in recent scikit-learn releases (this cell raises `TypeError` on 1.9). On Colab it works. On a newer local install, delete that argument: multinomial is already the default, and the results are identical.

In [ ]:
# #### EXERCISE 6 ####
#
# Quiz-style: return the bias in weight vector associated with logit of class=0
#
# hint: biases can be obtained from log_reg.intercept_ 
#       and the other coefficients from log_reg.coef_.


### Exercise 6: Multinomial Model Bias
**Objective.** Return in `lab_2.question_6()` the intercept for class 0 from the fitted multinomial logistic-regression model.

For class $k$, the model forms a score $s_k(x)=b_k+x^\top w_k$, where `log_reg.intercept_[k]` is $b_k$ and `log_reg.coef_[k]` stores $w_k$. The class probabilities are obtained by the softmax function, $P(y=k\mid x)=\frac{e^{s_k(x)}}{\sum_j e^{s_j(x)}}$. The intercept is the class score before feature contributions, so it helps define decision boundaries. The notebook fits `LogisticRegression` on the Iris training split and exposes the learned parameters for this quiz-style value.

### 🛠️ How to do Exercise 6

`log_reg.intercept_` is an array of shape `(3,)` holding $[b_0, b_1, b_2]$, and `log_reg.coef_` has shape `(3, 4)` (one row $\mathbf w_k$ per class). Class 0 (*setosa*) has bias `log_reg.intercept_[0]`. Run the helper below and put the number in `question_6()`:
```python
def question_6():
    return 8.68      # value of log_reg.intercept_[0] (verified ≈ 8.6786)
```
**Interpretation.**
- The intercepts were verified as $[8.68, 2.92, -11.60]$.
- Only *differences* between class scores matter in a softmax, because adding a constant to every $s_k$ leaves the probabilities unchanged. sklearn's regularised fit makes the parameters sum to ≈ 0 across classes.
- The large positive $b_0$ means *setosa* wins unless the (petal) features push the other scores up. Setosa has small petals, so its features contribute little to the competing scores.

In [ ]:
# Optional helper added for learning: inspect the multinomial parameters (Exercise 6)
print("intercepts b_k :", log_reg.intercept_)
print("coef_ shape    :", log_reg.coef_.shape, "(one weight vector per class)")
print("Bias for class 0 (answer to question_6):", log_reg.intercept_[0])

# Sanity check: softmax of the scores reproduces predict_proba
scores = testX @ log_reg.coef_.T + log_reg.intercept_
probs = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
print("softmax == predict_proba:", np.allclose(probs, log_reg.predict_proba(testX)))

In [ ]:
# In multinomial logistic regression, we have one score/logit per class
# the logit is obtained from its own weight vector.
# We propose here to visualise the score/logit map for each of the classes.

# the point of this cell is for you to visualise what the score maps do 
# just try different class ids and on different feature pairs.

classid = 2; # you can change this to 0,1,2

# Plot the risk score map (x'w_k) for the associated class weights vector w_k. 
featx = 1 # feat 0 will be on x-axis, you can change this to 0,1,2,3
featy = 2 # feat 1 will be on y-axis, you can change this to 0,1,2,3

# getting area of interest

x_min = trainX[:,featx].min()
x_max = trainX[:,featx].max()
y_min = trainX[:,featy].min()
y_max = trainX[:,featy].max()

# grid sampling the feature space
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 256), np.linspace(y_min, y_max, 256))

# making prediction for point on grid
Z = log_reg.coef_[classid,featx] * xx + yy * log_reg.coef_[classid,featy] 

# plot score map
plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, Z, levels=10, alpha=0.9)
# overlay dataset points
scatter = plt.scatter(trainX[:, featx], trainX[:, featy], c=(trainY==classid), alpha=0.8)

kw = scatter.legend_elements()
kw[1][0] = f"other classes"
kw[1][1] = f"class {classid} ({class_names[classid]})"
plt.legend(*kw)

plt.xlabel(f"Feature {featx} - {feature_names[featx]}")
plt.ylabel(f"Feature {featy} - {feature_names[featy]}")
plt.title(f"risk score map for class {classid} ({class_names[classid]})")
plt.show()




### 🔍 Interpreting the score maps

Each map colours the plane by $s_k=\mathbf x^\top\mathbf w_k$, restricted to two features. The bias is omitted, so only the *direction* of increase matters. The class is predicted wherever its score beats the others.
- The contours are **straight lines**, so multinomial logistic regression has **linear decision boundaries** (handout-02 slide 32).
- Try `classid = 0` with petal features `featx=2, featy=3`. The score increases towards small petals, which is exactly where the setosa points lie.

Non-linear boundaries need either hand-made features (Lab 1's polynomials) or **learned features** (hidden layers in Lab 4, convolutions in Lab 5).

In [ ]:
# #### EXERCISE 7 ####
# 
# In this exercise, you must find the best pair of features in the iris dataset.
#
# To find this pair, you will consider every possible pair and reduce the 
# input features to only that pair (and ignore the other two features). 
#
# hint:
# iterate through all possible pairs and for each pair:
# 1. modify both training and test sets to only include the two considered features
# 2. train a multinomial logistic regression model based on this reduced feature set
# 3. make prediction on the reduced test set 
# 4. report accuracy for that pair
#
# you can work in this notebook, but you need to report the best pair 
# and the accuracy in question_7 in lab_2.py

# You might need to do `importlib.reload(lab_1)` to reload labs_2 if you have
# modfied any functions in it.



### Exercise 7: Selecting an Iris Feature Pair
**Objective.** Find the best two-feature Iris representation and return its feature indices and test accuracy from `lab_2.question_7()`.

For every pair of the four columns, reduce both `trainX` and `testX` to that pair, fit a multinomial `LogisticRegression` model, predict the held-out labels, and compute $\operatorname{accuracy}=\frac{1}{n_{\mathrm{test}}}\sum_i\mathbb{1}[y_i=\hat{y}_i]$. Comparing the same metric across pairs identifies which two measurements retain the most discriminative information for this model. Using the fixed 70/30 split and `random_state=11` makes the comparison reproducible; evaluating on `testX`, rather than training accuracy, provides evidence of generalization and helps avoid selecting a pair only because it overfits the training examples.

### 🛠️ How to do Exercise 7 (feature-pair selection)

**Steps**
1. Enumerate the 6 pairs of feature indices with `itertools.combinations(range(4), 2)`.
2. For each pair, slice the columns `trainX[:, pair]` and `testX[:, pair]`.
3. Fit a **new** `LogisticRegression` with the **same settings** as above (`solver='newton-cg'`, `random_state=11`, multinomial). Predict on the reduced test set and compute the accuracy.
4. Report the pair with the highest **test** accuracy.

The optional helper cell below does exactly this.

**Verified results:**

| Pair | Features | Test accuracy |
|---|---|---|
| (0, 1) | sepal length, sepal width | 0.6889 |
| **(0, 2)** | **sepal length, petal length** | **0.9556** |
| (0, 3) | sepal length, petal width | 0.9111 |
| (1, 2) | sepal width, petal length | 0.9333 |
| (1, 3) | sepal width, petal width | 0.8667 |
| (2, 3) | petal length, petal width | 0.9333 |

```python
def question_7():
    return {'Features': (0, 2), 'Acc': 0.9556}   # 43/45 = 0.95556
```
**Interpretation.**
- Petal measurements carry most of the class information. Sepal-only (0,1) is poor because versicolor and virginica overlap heavily in sepal space.
- The best pair (0.956) even beats all four features (0.933) on this split. With only 45 test samples, one flower = 2.2%, so differences of one or two flowers are within noise.
- Choosing the pair by its **test** accuracy uses the test set for model selection, which makes the reported accuracy optimistic. In a real project you would select on a validation set or with cross-validation (handout-04 slides 34–37).

In [ ]:
# Optional helper added for learning: Exercise 7 search over all feature pairs
import itertools

results = {}
for pair in itertools.combinations(range(4), 2):
    cols = list(pair)
    clf = LogisticRegression(solver='newton-cg', random_state=11)   # multinomial is the default
    clf.fit(trainX[:, cols], trainY)
    acc = np.mean(clf.predict(testX[:, cols]) == testY)
    results[pair] = acc
    print(f"{pair} {feature_names[cols[0]]:>18} + {feature_names[cols[1]]:<18} accuracy = {acc:.4f}")

best = max(results, key=results.get)
print("\nBest pair:", best, "accuracy:", round(results[best], 4))
print("question_7 answer -> {'Features': %s, 'Acc': %.4f}" % (best, results[best]))

## ✅ Lab 2 key takeaways and self-check

**Takeaways**
1. Logistic regression = linear score + sigmoid/softmax → a probability. Its decision boundaries are linear.
2. Cross-entropy is the negative log-likelihood of the labels. Its gradient has the simple form $\frac1n X^\top(\mathbf p-\mathbf y)$.
3. With no closed form, we use **gradient descent**. The learning rate trades speed against stability, and feature scaling is essential.
4. A probability becomes a class through a **threshold**, and the threshold should reflect the costs of the errors.
5. On imbalanced data, accuracy hides poor minority-class performance. Lab 3 introduces better metrics.
6. Softmax + cross-entropy is the output layer of every classifier network in Labs 4–5.

**Self-check questions**
- Derive $\partial\mathcal L/\partial\mathbf w$ for logistic regression starting from $\sigma'(t)=\sigma(t)(1-\sigma(t))$.
- Why did gradient descent fail before balance was divided by 1000?
- What happens to the loss curve if $\eta$ is too large? Too small?
- Why is 97% accuracy *not* impressive on the Default data?
- Show that adding the same constant to every class score leaves the softmax unchanged.
- Why is selecting the feature pair on the test set slightly optimistic?